# Speicherformat für Bronze: NDJSON gegen Parquet

Dieses Notebook beantwortet eine Frage, die als Entscheid offen war, und liefert
die Messung für ein gefordertes Kapitel des Leistungsnachweises.

**Der Entscheid zu [Absprache 4.1](../docs/intern/projektabsprache.md)** lautet
wörtlich: *„Es wird angeschaut was performanter ist. Wenn Parquet besser ist,
wird es umgesetzt da es besser ist zu vereinheitlichen."* Genau das wird hier
angeschaut.

**LN1 verlangt** unter Storage „Evaluation, Wahl und Implementierung"
([Leistungsnachweise 4](../docs/intern/leistungsnachweise.md)). Implementiert ist
etwas, gewählt ist DuckDB — die Evaluation fehlte als Messung.

**Ausgangslage — Stand bis zum 11.09.2026.** Chicago, Census und Wetter legten
Bronze als Parquet ab. Detroit legte NDJSON ab, und diese eine Datei übernahm
dort beide Rollen: Sie war Beweisstück und abgeleitete Tabelle gleichzeitig. Die
Frage war also nicht nur, welches Format schneller ist, sondern ob Detroit auf
die Konvention der anderen drei nachgezogen werden soll.

> **Der Entscheid ist gefallen und umgesetzt.** Detroit schreibt seit dem
> 11.09.2026 Parquet und hat seit dem 14.09.2026 eine eigene Raw-Schicht. Dieses
> Notebook bleibt stehen, weil es die **Begründung** ist — und weil Abschnitt 9
> nachrechnet, ob seine Vorhersage eingetroffen ist. Alle Zeiten und Grössen
> unten sind neu gemessen, nicht die von damals.

## 1. Was verglichen wird — und was nicht

Verglichen werden **dieselben Daten in zwei Formaten**: Detroits Bronze-Lauf,
einmal als NDJSON wie die Quelle es ausliefert und einmal als Parquet. Beide
werden über DuckDB gelesen, weil das nach
[Absprache 5.3](../docs/intern/projektabsprache.md) der Abfrageweg ist.

> **Woher das NDJSON kommt.** Detroit schreibt seit dem 11.9. Parquet — das
> Ergebnis genau dieser Messung. Der alte NDJSON-Bestand ist gelöscht, damit
> niemand versehentlich darauf zugreift. Das Vergleichs-NDJSON entsteht
> deshalb zur Laufzeit aus der Spalte `_feature_json` des Parquet-Laufs: Sie
> bewahrt die geparste Antwort der Quelle auf. Die Messung vergleicht also
> weiterhin **dieselben Daten**, nur ohne eine 89-MB-Datei vorauszusetzen.

**Vier Vergleiche:**

1. Grösse auf der Platte, auch gegen gzip-komprimiertes NDJSON
2. Lesezeit für typische Abfragen
3. Ob `count(*)` die Daten überhaupt anfassen muss
4. Ob NDJSON sein Schema stabil errät

**Was die Messung nicht kann.** Sie läuft mit **warmem Betriebssystem-Cache**:
Die Datei liegt beim Messen schon im Arbeitsspeicher. Das nimmt den
Grössenunterschied aus der Rechnung und **unterschätzt damit Parquets
Vorteil** — bei kaltem Cache müsste NDJSON ein Vielfaches an Bytes von der
Platte holen. Ausserdem ist es eine Maschine und ein Datensatz; die
Grössenordnung ist belastbar, die zweite Dezimalstelle nicht.

In [ ]:
from pathlib import Path
import glob
import gzip
import os
import shutil
import statistics
import tempfile
import time

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'bronze').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data' / 'bronze').exists():
    raise RuntimeError('Projekt-Root mit data/bronze nicht gefunden')

laeufe = sorted(glob.glob(str(
    PROJECT_ROOT / 'data' / 'bronze' / 'city=detroit' / '**' / 'run_*' / 'part-0001.parquet'
), recursive=True))
if not laeufe:
    raise RuntimeError(
        'Kein Detroit-Bronze-Lauf gefunden. Zuerst: python -m '
        'detroit_pipeline.extract_detroit --mode delta --years 2024'
    )
QUELLLAUF = laeufe[-1]

ARBEIT = Path(tempfile.mkdtemp(prefix='storage-eval-'))
NDJSON = str(ARBEIT / 'detroit.ndjson')

# _feature_json bewahrt die Antwort der Quelle unveraendert auf. Eine Zeile je
# Datensatz ergibt genau das NDJSON, das die Pipeline frueher geschrieben hat.
con = duckdb.connect()
con.execute(f"""
    COPY (SELECT _feature_json FROM read_parquet('{QUELLLAUF}'))
    TO '{NDJSON}' (FORMAT CSV, HEADER false, QUOTE '', DELIMITER E'\t')
""")

zeilen = con.execute(f"SELECT count(*) FROM read_json_auto('{NDJSON}')").fetchone()[0]
erwartet = con.execute(f"SELECT count(*) FROM read_parquet('{QUELLLAUF}')").fetchone()[0]
assert zeilen == erwartet, f'{zeilen} statt {erwartet} Zeilen rekonstruiert'

print('DuckDB      :', duckdb.__version__)
print('Quelllauf   :', Path(QUELLLAUF).parent.name)
print('Zeilen      :', f'{zeilen:,}')
print('NDJSON      :', f'{os.path.getsize(NDJSON) / 1e6:.1f} MB  (rekonstruiert)')


### Dieselben Daten in die Vergleichsformate schreiben

Umgewandelt wird aus dem NDJSON selbst, nicht aus einer zweiten Quelle — sonst
verglichen wir zwei Datenbestände statt zwei Formate.

In [ ]:
# ARBEIT steht schon aus der vorigen Zelle - dort liegt das rekonstruierte
# NDJSON, das hier in die Vergleichsformate geschrieben wird.
print('Arbeitsverzeichnis:', ARBEIT)

formate = {}
for name, optionen in (
    ('Parquet snappy', 'FORMAT PARQUET, COMPRESSION SNAPPY'),
    ('Parquet zstd', 'FORMAT PARQUET, COMPRESSION ZSTD'),
):
    ziel = ARBEIT / f"detroit.{name.split()[-1]}.parquet"
    con.execute(
        f"COPY (SELECT * FROM read_json_auto('{NDJSON}')) TO '{ziel}' ({optionen})"
    )
    formate[name] = ziel

gz = ARBEIT / 'detroit.ndjson.gz'
with open(NDJSON, 'rb') as quelle, gzip.open(gz, 'wb', compresslevel=6) as ziel:
    shutil.copyfileobj(quelle, ziel)

PARQUET = formate['Parquet snappy']
print('geschrieben:', ', '.join(p.name for p in [*formate.values(), gz]))

## 2. Grösse auf der Platte

Der Vergleich mit **gzip-komprimiertem NDJSON** gehört dazu, sonst vergleicht
man komprimiert gegen unkomprimiert und das Ergebnis sagt nur, dass Kompression
wirkt.

In [ ]:
kandidaten = [
    ('NDJSON, wie ausgeliefert', Path(NDJSON)),
    ('NDJSON + gzip', gz),
    ('Parquet snappy', formate['Parquet snappy']),
    ('Parquet zstd', formate['Parquet zstd']),
]
basis = os.path.getsize(NDJSON)
print(f"{'Format':<26}{'Groesse':>10}{'vs NDJSON':>12}{'vs gzip':>10}")
gzip_groesse = os.path.getsize(gz)
for label, pfad in kandidaten:
    groesse = os.path.getsize(pfad)
    print(f'{label:<26}{groesse / 1e6:>7.1f} MB{groesse / basis:>11.1%}'
          f'{groesse / gzip_groesse:>9.1%}')

## 3. Lesezeiten

Fünf Abfragen, jede siebenmal, gemeldet wird der Median. Die letzte Abfrage
materialisiert alle Spalten — das ist der Fall, in dem Parquets Vorteil am
kleinsten ist, und er gehört mitgemessen.

In [ ]:
def median_zeit(sql, laeufe=7):
    zeiten = []
    for _ in range(laeufe):
        start = time.perf_counter()
        con.execute(sql).fetchall()
        zeiten.append(time.perf_counter() - start)
    return statistics.median(zeiten)


quellen = {
    'NDJSON': f"read_json_auto('{NDJSON}')",
    'Parquet': f"read_parquet('{PARQUET}')",
}
abfragen = {
    'Zeilen zaehlen': 'SELECT count(*) FROM {q}',
    'eine Spalte lesen': 'SELECT count(attributes.offense_category) FROM {q}',
    'gefiltert zaehlen':
        "SELECT count(*) FROM {q} WHERE attributes.offense_category = 'ASSAULT'",
    'nach Kategorie gruppieren':
        'SELECT attributes.offense_category, count(*) FROM {q} GROUP BY 1',
    'fuenf Spalten, alle Zeilen':
        '''SELECT attributes.incident_entry_id, attributes.offense_category,
                  attributes.incident_occurred_at, geometry.x, geometry.y
           FROM {q}''',
    'alle Spalten materialisieren': 'SELECT * FROM {q}',
}

print(f"{'Abfrage':<32}{'NDJSON':>10}{'Parquet':>10}{'Faktor':>9}")
ergebnisse = {}
for label, muster in abfragen.items():
    a = median_zeit(muster.format(q=quellen['NDJSON']))
    b = median_zeit(muster.format(q=quellen['Parquet']))
    ergebnisse[label] = (a, b)
    # Unter zehn eine Dezimalstelle, sonst wird 1,25 zu "1x" gerundet.
    verhaeltnis = a / b if b > 0 else None
    if verhaeltnis is None:
        faktor = 'n/a'
    else:
        faktor = f'{verhaeltnis:.1f}x' if verhaeltnis < 10 else f'{verhaeltnis:.0f}x'
    print(f'{label:<32}{a:>9.3f}s{b:>9.3f}s{faktor:>9}')

**Die Zeile, die für das Projekt zählt, ist „fünf Spalten, alle Zeilen".** Genau
das wird die Silver-Tabelle `crime_offense` tun: wenige Felder aus jeder Zeile
lesen, nicht alles materialisieren.

NDJSONs Zeiten sind über die ersten vier Abfragen nahezu konstant — und das ist
der Kern: **JSON kennt keine Spaltenauswahl.** Wer ein Feld braucht, parst die
ganze Datei. Parquet liest nur die angefragten Spalten.

Beim vollständigen Materialisieren schrumpft der Vorsprung deutlich, weil dort
das Aufbauen der Ergebniszeilen dominiert und nicht das Lesen.

## 4. Muss `count(*)` die Daten anfassen?

Parquet führt die Zeilenzahl im Dateikopf mit. Der grosse Faktor oben ist
deshalb kein Geschwindigkeitsvorteil beim Lesen, sondern der Umstand, dass
**gar nicht gelesen wird**. Der Abfrageplan zeigt es.

In [ ]:
for label, quelle in quellen.items():
    plan = con.execute(f'EXPLAIN SELECT count(*) FROM {quelle}').fetchone()[1]
    schritte = [
        zeile.strip(' │')
        for zeile in plan.splitlines()
        if any(wort in zeile for wort in ('SCAN', 'READ', 'AGGREGATE'))
    ]
    print(f'{label:<9}', ' -> '.join(dict.fromkeys(schritte)) or plan.splitlines()[0])

`COLUMN_DATA_SCAN` heisst: aus den Metadaten. NDJSON muss dafür die Datei lesen.

Für die Vollständigkeitsnachweise der Pipelines ist das nicht nebensächlich —
jede Prüfung „so viele Zeilen wie erwartet" ist bei Parquet kostenlos.

## 5. Errät NDJSON sein Schema stabil?

Das war die Sorge, die gegen NDJSON sprechen könnte: DuckDB leitet die Typen
aus einer Stichprobe ab. Rät es bei kleiner Stichprobe anders als bei grosser,
wandert das Schema von Lauf zu Lauf.

**Geprüft statt vermutet:** drei Stichprobengrössen, einmal 100 Zeilen, einmal
2'048 und einmal die ganze Datei.

In [ ]:
schemata = {}
for stichprobe in (100, 2048, -1):
    typ = con.execute(
        f"DESCRIBE SELECT attributes FROM read_json_auto('{NDJSON}', "
        f'sample_size={stichprobe})'
    ).fetchone()[1]
    schemata[stichprobe] = str(typ)
    beschriftung = 'ganze Datei' if stichprobe == -1 else f'{stichprobe} Zeilen'
    print(f'{beschriftung:<14} {len(str(typ)):>5} Zeichen Typbeschreibung')

print()
print('verschiedene Schemata:', len(set(schemata.values())))
print('Ergebnis:', 'stabil' if len(set(schemata.values())) == 1 else 'INSTABIL')

**Die Sorge trifft für diesen Datensatz nicht zu.** Alle drei Stichproben
ergeben dasselbe Schema. Das ist ein Argument *für* NDJSON, und es gehört
genannt — eine Evaluation, die nur die Punkte aufzählt, die der Vorschlag
stützt, ist keine.

Der Grund ist plausibel: ArcGIS liefert für jede Zeile dieselben Felder, leere
Werte als `null` statt als fehlendes Feld. Bei einer Quelle mit optionalen
Feldern wäre das Ergebnis ein anderes.

## 6. Was das für Detroit konkret heisst

Damals übernahm Detroits NDJSON beide Rollen. Die anderen drei Quellen trennen:
Raw hält das Beweisstück, Bronze die abgeleitete Tabelle. Nachziehen hiess also
nicht „NDJSON durch Parquet ersetzen", sondern **eine Schicht ergänzen** — und
das Verblüffende war, dass beide Schichten zusammen deutlich kleiner ausfallen
als die eine davor. Was daraus wurde, rechnet Abschnitt 9 nach.

In [ ]:
heute = os.path.getsize(NDJSON)
raw_neu = os.path.getsize(gz)
bronze_neu = os.path.getsize(PARQUET)

print('heute   eine Datei fuer beide Rollen')
print(f'          Bronze NDJSON            {heute / 1e6:>7.1f} MB')
print()
print('nachgezogen, wie Chicago und Census')
print(f'          Raw    NDJSON + gzip     {raw_neu / 1e6:>7.1f} MB')
print(f'          Bronze Parquet snappy    {bronze_neu / 1e6:>7.1f} MB')
print(f'          zusammen                 {(raw_neu + bronze_neu) / 1e6:>7.1f} MB')
print()
ersparnis = 1 - (raw_neu + bronze_neu) / heute
print(f'{ersparnis:.0%} weniger Platz - bei einer Schicht mehr.')

## 7. Antwort auf Absprache 4.1

**Parquet ist besser.** Der Entscheid sagt, dass es dann umgesetzt wird.

| Kriterium | Ergebnis |
| --- | --- |
| Grösse | Parquet ist rund 14-mal kleiner als NDJSON, rund 1,7-mal kleiner als gzip-komprimiertes NDJSON |
| Realistische Abfrage | wenige Spalten über alle Zeilen: **rund 5-mal schneller** |
| Zeilenzahl | aus den Metadaten, ohne die Daten zu lesen |
| Vollständiges Materialisieren | nur noch etwa 1,3-mal schneller |
| Schema-Stabilität | **kein Unterschied** — NDJSON rät hier stabil |
| Einheitlichkeit | Chicago, Census und Wetter liegen schon als Parquet |

**Der stärkste Einzelbefund** ist nicht die Geschwindigkeit, sondern die
Kombination: Detroit kann die fehlende Raw-Schicht bekommen **und** dabei 81 %
Platz sparen. Damit fällt das Argument weg, die Trennung sei zu teuer. Wie viel
davon in der Umsetzung übrig blieb, steht in Abschnitt 9.

**Was gegen Parquet spricht**, damit es genannt ist: Es ist nicht lesbar ohne
Werkzeug. Ein `head` auf die NDJSON-Datei zeigt eine Zeile; für Parquet braucht
es DuckDB oder pyarrow. Für die Fehlersuche an einer neuen Quelle ist NDJSON
angenehmer — und genau deshalb bleibt es in Raw.

**Folgeentscheid.** [Absprache 4.2](../docs/intern/projektabsprache.md) („Angleichung
Detroits") hatte als Entscheid „Hängt von 4.1 ab". Diese Abhängigkeit ist
hiermit aufgelöst.

## 8. Grenzen dieser Messung

- **Warmer Cache.** Die Datei liegt beim Messen im Arbeitsspeicher. Das nimmt
  den Grössenunterschied aus der Rechnung und unterschätzt Parquet.
- **Eine Maschine, ein Datensatz.** Die Grössenordnungen sind belastbar, exakte
  Faktoren nicht übertragbar.
- **Nur das Bronze-Format.** Die Wahl der Speicherumgebung — Dateien plus
  DuckDB gegen PostgreSQL/PostGIS gegen Cloud — steht in
  [Leistungsnachweise 4](../docs/intern/leistungsnachweise.md) und ist eine andere
  Frage als die hier gemessene.
- **Kein Test des Schreibpfads.** Gemessen wurde Lesen. Die Ingestion schreibt
  einmal je Lauf, dort fällt der Unterschied kaum ins Gewicht.

## 9. Was daraus geworden ist

Abschnitt 6 war ein **Modell**: Raw als gzip-komprimiertes NDJSON, Bronze als
Parquet ohne Zugaben. Umgesetzt wurde beides etwas anders. Raw hält die
HTTP-Antworten als gezippte JSON-Seiten, Seite für Seite wie geliefert. Und
Bronze führt zusätzlich die Spalte `_feature_json` mit dem geparsten
Quellfeature.

Eine Vorhersage, die niemand nachrechnet, ist keine. Die Zelle vergleicht das
Modell mit dem, was wirklich auf der Platte liegt.

In [ ]:
ist_bronze = Path(QUELLLAUF).parent
ist_raw = PROJECT_ROOT / 'data' / 'raw' / ist_bronze.relative_to(
    PROJECT_ROOT / 'data' / 'bronze')

ndjson_gross = os.path.getsize(NDJSON)
modell_raw, modell_bronze = os.path.getsize(gz), os.path.getsize(PARQUET)
ist_raw_gross = sum(f.stat().st_size for f in ist_raw.glob('*.json.gz'))
ist_bronze_gross = sum(f.stat().st_size for f in ist_bronze.glob('*.parquet'))

ohne = ARBEIT / 'ohne-feature-json.parquet'
con.execute(f"COPY (SELECT * EXCLUDE(_feature_json) FROM read_parquet('{QUELLLAUF}')) "
            f"TO '{ohne}' (FORMAT PARQUET, COMPRESSION SNAPPY)")

print(f'{"":12}{"Modell (Abschnitt 6)":>22}{"umgesetzt":>16}')
for label, modell, ist in (('Raw', modell_raw, ist_raw_gross),
                           ('Bronze', modell_bronze, ist_bronze_gross),
                           ('zusammen', modell_raw + modell_bronze,
                            ist_raw_gross + ist_bronze_gross)):
    print(f'{label:12}{modell / 1e6:>19.1f} MB{ist / 1e6:>13.1f} MB')

print()
print(f'gegenueber {ndjson_gross / 1e6:.1f} MB NDJSON von damals:')
print(f'  vorhergesagt  {1 - (modell_raw + modell_bronze) / ndjson_gross:.0%} weniger')
print(f'  erreicht      {1 - (ist_raw_gross + ist_bronze_gross) / ndjson_gross:.0%} weniger')
print()
print(f'Die Luecke steckt in Bronze: ohne _feature_json waeren es '
      f'{os.path.getsize(ohne) / 1e6:.1f} MB statt {ist_bronze_gross / 1e6:.1f} MB.')


**Die Richtung stimmte, die Höhe nicht ganz.** Vorhergesagt waren 81 % weniger
Platz, erreicht sind rund 70 %. Der Unterschied ist kein Messfehler, sondern
eine Entscheidung, die nach dieser Evaluation fiel: Bronze behielt
`_feature_json`.

Damit liegt die Antwort der Quelle **zweimal** auf der Platte — einmal als
HTTP-Bytes in Raw, einmal geparst in Bronze. Das ist Absicht und steht so im
Modulkopf von `extract_detroit`: Das geparste Feature „genügt für die
Auswertung, ist aber kein Beweis". Raw liefert den Beweis, `_feature_json` die
Bequemlichkeit — dieses Notebook selbst baut sein Vergleichs-NDJSON daraus.

Die 11 MB sind der Preis dafür. Ob er sich lohnt, ist eine Frage für später;
hier gehört sie genannt, weil die 81 % sonst als erreicht gelesen würden.

In [ ]:
shutil.rmtree(ARBEIT, ignore_errors=True)
con.close()
print('Arbeitsverzeichnis entfernt:', ARBEIT)